<a href="https://colab.research.google.com/github/asokone/Advanced-Python-and-Machine-Learning/blob/main/IBM%20SkillsBuild%20AI%20Fundamentals%3A%20Machine%20learning%20with%20Python%20-%20Natural%20Language%20Processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## IBM SkillsBuild AI Fundamentals: Machine learning with Python - Natural Language Processing

https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment-latest
Twitter-roBERTa-base for Sentiment Analysis - UPDATED (2022)
This is a RoBERTa-base model trained on ~124M tweets from January 2018 to December 2021, and finetuned for sentiment analysis with the TweetEval benchmark. The original Twitter-based RoBERTa model can be found here  https://huggingface.co/cardiffnlp/twitter-roberta-base-2021-124m and the original reference paper is TweetEval. This model is suitable for English.
•	Reference Paper: TimeLMs paper.
•	Git Repo: TimeLMs official repository.


https://huggingface.co/cardiffnlp/twitter-roberta-base-2021-124m

Twitter 2021 124M (RoBERTa-base)
This is a RoBERTa-base model trained on 123.86M tweets until the end of 2021. More details and performance scores are available in the TimeLMs paper.
Below, we provide some usage examples using the standard Transformers interface. For another interface more suited to comparing predictions and perplexity scores between models trained at different temporal intervals, check the TimeLMs repository.
For other models trained until different periods, check this table.

Preprocess Text
Replace usernames and links for placeholders: "@user" and "http". If you're interested in retaining verified users which were also retained during training, you may keep the users listed here.


In [ ]:
def preprocess(text):
    preprocessed_text = []
    for t in text.split():
        if len(t) > 1:
            t = '@user' if t[0] == '@' and t.count('@') == 1 else t
            t = 'http' if t.startswith('http') else t
        preprocessed_text.append(t)
    return ' '.join(preprocessed_text)

Example Masked Language Model


In [ ]:
from transformers import pipeline, AutoTokenizer

MODEL = "cardiffnlp/twitter-roberta-base-2021-124m"
fill_mask = pipeline("fill-mask", model=MODEL, tokenizer=MODEL)
tokenizer = AutoTokenizer.from_pretrained(MODEL)

def pprint(candidates, n):
    for i in range(n):
        token = tokenizer.decode(candidates[i]['token'])
        score = candidates[i]['score']
        print("%d) %.5f %s" % (i+1, score, token))

texts = [
    "So glad I'm <mask> vaccinated.",
    "I keep forgetting to bring a <mask>.",
    "Looking forward to watching <mask> Game tonight!",
]

for text in texts:
    t = preprocess(text)
    print(f"{'-'*30}\n{t}")
    candidates = fill_mask(t)
    pprint(candidates, 5)


Example Tweet Embeddings

In [ ]:
from transformers import AutoTokenizer, AutoModel
import numpy as np
from scipy.spatial.distance import cosine
from collections import Counter

def get_embedding(text):  # naive approach for demonstration
  text = preprocess(text)
  encoded_input = tokenizer(text, return_tensors='pt')
  features = model(**encoded_input)
  features = features[0].detach().cpu().numpy()
  return np.mean(features[0], axis=0)


MODEL = "cardiffnlp/twitter-roberta-base-2021-124m"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModel.from_pretrained(MODEL)

query = "The book was awesome"
tweets = ["I just ordered fried chicken 🐣",
          "The movie was great",
          "What time is the next game?",
          "Just finished reading 'Embeddings in NLP'"]

sims = Counter()
for tweet in tweets:
    sim = 1 - cosine(get_embedding(query), get_embedding(tweet))
    sims[tweet] = sim

print('Most similar to: ', query)
print(f"{'-'*30}")
for idx, (tweet, sim) in enumerate(sims.most_common()):
    print("%d) %.5f %s" % (idx+1, sim, tweet))

Example Feature Extraction


In [ ]:
from transformers import AutoTokenizer, AutoModel
import numpy as np

MODEL = "cardiffnlp/twitter-roberta-base-2021-124m"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

text = "Good night 😊"
text = preprocess(text)

# Pytorch
model = AutoModel.from_pretrained(MODEL)
encoded_input = tokenizer(text, return_tensors='pt')
features = model(**encoded_input)
features = features[0].detach().cpu().numpy()
features_mean = np.mean(features[0], axis=0)
#features_max = np.max(features[0], axis=0)

# # Tensorflow
# model = TFAutoModel.from_pretrained(MODEL)
# encoded_input = tokenizer(text, return_tensors='tf')
# features = model(encoded_input)
# features = features[0].numpy()
# features_mean = np.mean(features[0], axis=0)
# #features_max = np.max(features[0], axis=0)

https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment-latest

Twitter-roBERTa-base for Sentiment Analysis - UPDATED (2022)
This is a RoBERTa-base model trained on ~124M tweets from January 2018 to December 2021, and finetuned for sentiment analysis with the TweetEval benchmark. The original Twitter-based RoBERTa model can be found here and the original reference paper is TweetEval. This model is suitable for English.

Reference Paper: TimeLMs paper.
Git Repo: TimeLMs official repository.
Labels: 0 -> Negative; 1 -> Neutral; 2 -> Positive

This sentiment analysis model has been integrated into TweetNLP. You can access the demo here.

Example Pipeline

In [ ]:
from transformers import pipeline
model_path = "cardiffnlp/twitter-roberta-base-sentiment-latest"
sentiment_task = pipeline("sentiment-analysis", model=model_path, tokenizer=model_path)
sentiment_task("Covid cases are increasing fast!")

Full classification example

In [ ]:
from transformers import AutoModelForSequenceClassification
from transformers import AutoTokenizer, AutoConfig
import numpy as np
from scipy.special import softmax

MODEL = f"cardiffnlp/twitter-roberta-base-sentiment-latest"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
config = AutoConfig.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL)


In [ ]:
def analyze_sentiment(text):
    text_preprocessed = preprocess(text)
    encoded_input = tokenizer(text_preprocessed, return_tensors='pt')
    output = model(**encoded_input)
    scores = output[0][0].detach().numpy()
    scores = softmax(scores)

    # Print labels and scores
    ranking = np.argsort(scores)
    ranking = ranking[::-1]
    results = []
    for i in range(scores.shape[0]):
        label = config.id2label[ranking[i]]
        score = scores[ranking[i]]
        results.append({"label": label, "score": float(f"{score:.4f}")})
    return results

In [ ]:
sample_text = "This is such a fantastic day!"
sentiment_results = analyze_sentiment(sample_text)
print(f"Sentiment analysis for '{sample_text}':")
for res in sentiment_results:
    print(f"  {res['label']}: {res['score']:.4f}")

sample_text_2 = "I am not happy with this situation."
sentiment_results_2 = analyze_sentiment(sample_text_2)
print(f"\nSentiment analysis for '{sample_text_2}':")
for res in sentiment_results_2:
    print(f"  {res['label']}: {res['score']:.4f}")

In [ ]:
sentences_to_analyze = [
    "I absolutely love this new product! It's fantastic and works perfectly.",
    "This movie was just okay. I neither loved nor hated it.",
    "I am so disappointed with the service I received. It was terrible.",
    "The weather today is surprisingly pleasant.",
    "What a frustrating experience, I will not be returning."
]

Now that we have a list of sentences, let's process them in bulk using our `analyze_sentiment` function.

In [ ]:
print("Bulk Sentiment Analysis Results:")
for sentence in sentences_to_analyze:
    results = analyze_sentiment(sentence)
    print(f"\nSentence: '{sentence}'")
    for res in results:
        print(f"  {res['label']}: {res['score']:.4f}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Prepare data for visualization
sentiment_data = []
for sentence in sentences_to_analyze:
    results = analyze_sentiment(sentence)
    # Get the sentiment with the highest score (primary sentiment)
    primary_sentiment = results[0]
    for res in results:
        if res['score'] > primary_sentiment['score']:
            primary_sentiment = res
    sentiment_data.append({
        'sentence': sentence,
        'sentiment': primary_sentiment['label'],
        'score': primary_sentiment['score']
    })

df_sentiment = pd.DataFrame(sentiment_data)

Now, let's visualize these primary sentiment scores using a bar chart.

In [ ]:
plt.figure(figsize=(12, 7))
sns.barplot(x='sentence', y='score', hue='sentiment', data=df_sentiment, palette={'positive': 'green', 'neutral': 'blue', 'negative': 'red'})
plt.title('Primary Sentiment Scores for Sentences')
plt.xlabel('Sentence')
plt.ylabel('Sentiment Score')
plt.xticks(rotation=45, ha='right')
plt.ylim(0, 1) # Scores are between 0 and 1
plt.tight_layout()
plt.show()

In [ ]:
mean_sentiment_scores = df_sentiment.groupby('sentiment')['score'].mean().reset_index()
print("Mean sentiment scores per category:")
display(mean_sentiment_scores)